# 01 - Register an ADLS manifest event

This fast intake notebook is called by the event-triggered pipeline. It deduplicates the CloudEvent, validates the manifest-last producer contract, and inserts one `QUEUED` work row. It never performs video inference.

**After importing into Fabric:** On the configuration code cell, select **... -> Toggle parameter cell** and confirm the parameter indicator. Then attach and pin `people_counter_<environment>` as this notebook's default Lakehouse.

## Event parameters

The `pc-event-intake` pipeline normally injects the values below. Running this notebook interactively leaves the defaults empty and the registration cell will fail intentionally.

For an interactive smoke test, copy `source`, `id`, `type`, `time`, and `subject` from one Eventstream preview event. Set `MANIFEST_URI` to `data.destinationUrl` and give `PIPELINE_RUN_ID` a manual correlation value. Rerun the parameter cell before running the registration cell. The destination JSON must contain the complete manifest-version-1 contract.

In [ ]:
EVENT_SOURCE = ""
EVENT_ID = ""
EVENT_TYPE = "Microsoft.Storage.BlobRenamed"
EVENT_TIME = ""
SUBJECT = ""
MANIFEST_URI = ""
PIPELINE_RUN_ID = ""
DATABASE = ""
TABLE_PREFIX = "people_counter"
MAX_ATTEMPTS = 4
PRIORITY = 100
PIPELINE = "rtdetr-osnet"
DEVICE_VARIANT = "cpu"
DEVICE = "cpu"
BATCH_SIZE = 1
SAMPLE_FPS = 3.0
DETECTION_THRESHOLD = 0.6
USE_FP16 = False
DETECTOR_MODEL = "r18"
CAMERA_MOTION_COMPENSATION = None

In [ ]:
from datetime import datetime, timedelta, timezone
from typing import Any
from urllib.parse import urlsplit, urlunsplit
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError
import hashlib
import json
import math
import re

from delta.tables import DeltaTable
import notebookutils
from pyspark.sql import SparkSession, functions as F


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
SHA256 = re.compile(r"^[0-9a-f]{64}$")


required_event_parameters = {
    "EVENT_SOURCE": EVENT_SOURCE,
    "EVENT_ID": EVENT_ID,
    "EVENT_TYPE": EVENT_TYPE,
    "EVENT_TIME": EVENT_TIME,
    "SUBJECT": SUBJECT,
    "MANIFEST_URI": MANIFEST_URI,
}
missing_event_parameters = sorted(
    name
    for name, value in required_event_parameters.items()
    if not isinstance(value, str) or not value.strip()
)
if missing_event_parameters:
    raise ValueError(
        "Missing pipeline event parameters: "
        f"{', '.join(missing_event_parameters)}. "
        "Run this notebook through pc-event-intake, or populate and rerun "
        "the tagged parameter cell for an interactive smoke test."
    )


def require_text(value: object, name: str) -> str:
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{name} must be a non-empty string")
    return value.strip()


def parse_bool(value: object, name: str) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
        return value.strip().lower() == "true"
    raise ValueError(f"{name} must be true or false")


def parse_optional_bool(value: object, name: str) -> bool | None:
    if value is None or (isinstance(value, str) and value.strip().lower() in {"", "null", "none"}):
        return None
    return parse_bool(value, name)


def parse_utc(value: object, name: str) -> datetime:
    text = require_text(value, name)
    parsed = datetime.fromisoformat(text.replace("Z", "+00:00"))
    if parsed.tzinfo is None:
        raise ValueError(f"{name} must include an offset or Z")
    return parsed.astimezone(timezone.utc)


def normalize_uri(value: object, name: str) -> str:
    text = require_text(value, name)
    parts = urlsplit(text)
    if parts.scheme == "https" and parts.netloc.endswith((".blob.core.windows.net", ".dfs.core.windows.net")):
        account = parts.netloc.split(".", 1)[0]
        path_parts = [part for part in parts.path.split("/") if part]
        if len(path_parts) < 2:
            raise ValueError(f"{name} storage URL must include a container and path")
        container, *blob_path = path_parts
        return f"abfss://{container}@{account}.dfs.core.windows.net/{'/'.join(blob_path)}"
    if parts.scheme not in {"abfs", "abfss"} or not parts.netloc:
        raise ValueError(f"{name} must be an ADLS abfs URI or Azure Blob HTTPS URL")
    normalized_path = "/" + "/".join(part for part in parts.path.split("/") if part)
    return urlunsplit((parts.scheme.lower(), parts.netloc.lower(), normalized_path, "", ""))


def parse_manifest(payload: dict[str, Any], manifest_uri: str) -> dict[str, Any]:
    if payload.get("schema_version") != 1:
        raise ValueError("Unsupported manifest schema_version; expected 1")
    required = (
        "asset_id",
        "asset_version",
        "video_uri",
        "source_etag",
        "expected_size_bytes",
        "expected_sha256",
        "camera_id",
        "location_id",
        "captured_at_utc",
        "camera_timezone",
        "counting_line",
    )
    missing = sorted(set(required) - set(payload))
    if missing:
        raise ValueError(f"Manifest is missing required fields: {missing}")
    values = {
        field: require_text(payload[field], field)
        for field in required
        if field not in {"expected_size_bytes", "counting_line"}
    }
    video_uri = normalize_uri(values["video_uri"], "video_uri")
    if "/incoming/" not in urlsplit(video_uri).path or video_uri.endswith(".json"):
        raise ValueError("video_uri must reference a video under incoming/")
    size = payload["expected_size_bytes"]
    if isinstance(size, bool) or not isinstance(size, int) or size <= 0:
        raise ValueError("expected_size_bytes must be a positive integer")
    expected_sha256 = values["expected_sha256"].lower()
    if SHA256.fullmatch(expected_sha256) is None:
        raise ValueError("expected_sha256 must be 64 lowercase hexadecimal characters")
    line = payload["counting_line"]
    if (
        not isinstance(line, list)
        or len(line) != 4
        or any(isinstance(value, bool) or not isinstance(value, int) for value in line)
    ):
        raise ValueError("counting_line must contain exactly four integers")
    try:
        ZoneInfo(values["camera_timezone"])
    except ZoneInfoNotFoundError as error:
        raise ValueError("camera_timezone must be a valid IANA timezone") from error
    duration = payload.get("duration_seconds")
    if duration is not None and (
        isinstance(duration, bool)
        or not isinstance(duration, (int, float))
        or not math.isfinite(duration)
        or duration <= 0
    ):
        raise ValueError("duration_seconds must be positive when supplied")
    return {
        "asset_id": values["asset_id"],
        "asset_version": values["asset_version"],
        "source_uri": video_uri,
        "manifest_uri": manifest_uri,
        "source_etag": values["source_etag"],
        "expected_size_bytes": size,
        "expected_sha256": expected_sha256,
        "camera_id": values["camera_id"],
        "location_id": values["location_id"],
        "captured_at_utc": parse_utc(values["captured_at_utc"], "captured_at_utc"),
        "camera_timezone": values["camera_timezone"],
        "duration_seconds": float(duration) if duration is not None else None,
        "counting_line": line,
    }


def canonical_config(counting_line: list[int]) -> tuple[str, str]:
    pipeline = require_text(PIPELINE, "PIPELINE")
    if pipeline not in {"rtdetr-osnet", "rfdetr-botsort"}:
        raise ValueError("PIPELINE must be rtdetr-osnet or rfdetr-botsort")
    device_variant = require_text(DEVICE_VARIANT, "DEVICE_VARIANT")
    if device_variant != "cpu":
        raise ValueError("Fabric-native Spark requires DEVICE_VARIANT=cpu")
    device = require_text(DEVICE, "DEVICE")
    use_fp16 = parse_bool(USE_FP16, "USE_FP16")
    if device != "cpu" or use_fp16:
        raise ValueError("Fabric-native CPU execution requires DEVICE=cpu and USE_FP16=false")
    batch_size = int(BATCH_SIZE)
    if batch_size < 1:
        raise ValueError("BATCH_SIZE must be at least 1")
    sample_fps = None if SAMPLE_FPS is None else float(SAMPLE_FPS)
    if sample_fps is not None and sample_fps <= 0:
        raise ValueError("SAMPLE_FPS must be positive or null")
    threshold = float(DETECTION_THRESHOLD)
    if not 0 <= threshold <= 1:
        raise ValueError("DETECTION_THRESHOLD must be between 0 and 1")
    if DETECTOR_MODEL not in {"r18", "r50"}:
        raise ValueError("DETECTOR_MODEL must be r18 or r50")
    value = {
        "pipeline": pipeline,
        "device_variant": device_variant,
        "device": device,
        "batch_size": batch_size,
        "sample_fps": sample_fps,
        "detection_threshold": threshold,
        "use_fp16": use_fp16,
        "line": counting_line,
        "detector_model": DETECTOR_MODEL,
        "camera_motion_compensation": parse_optional_bool(
            CAMERA_MOTION_COMPENSATION,
            "CAMERA_MOTION_COMPENSATION",
        ),
    }
    encoded = json.dumps(value, sort_keys=True, separators=(",", ":"))
    return encoded, hashlib.sha256(encoded.encode("utf-8")).hexdigest()


database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")
if isinstance(MAX_ATTEMPTS, bool) or int(MAX_ATTEMPTS) < 1:
    raise ValueError("MAX_ATTEMPTS must be at least 1")


def table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
receipts_table = table("event_receipts")
work_table = table("video_work")
registration_leases_table = table("registration_leases")
event_source = require_text(EVENT_SOURCE, "EVENT_SOURCE")
event_id = require_text(EVENT_ID, "EVENT_ID")
event_key = hashlib.sha256(f"{event_source}\n{event_id}".encode("utf-8")).hexdigest()
event_type = require_text(EVENT_TYPE, "EVENT_TYPE")
event_time = parse_utc(EVENT_TIME, "EVENT_TIME")
subject = require_text(SUBJECT, "SUBJECT")
manifest_uri = normalize_uri(MANIFEST_URI, "MANIFEST_URI")
if "/incoming/" not in urlsplit(manifest_uri).path or not manifest_uri.endswith(".json"):
    raise ValueError("MANIFEST_URI must be an incoming/*.json path")
now = datetime.now(timezone.utc)
lock_source = spark_session.createDataFrame(
    [("global", event_key, now, now + timedelta(minutes=5))],
    "lock_name string, owner_id string, acquired_at timestamp, expires_at timestamp",
)
(
    DeltaTable.forName(spark_session, registration_leases_table)
    .alias("t")
    .merge(lock_source.alias("s"), "t.lock_name = s.lock_name")
    .whenMatchedUpdateAll(condition="t.expires_at <= current_timestamp() OR t.owner_id = s.owner_id")
    .execute()
)
lock_rows = spark_session.table(registration_leases_table).where(F.col("lock_name") == "global").limit(2).collect()
if len(lock_rows) != 1 or lock_rows[0].owner_id != event_key:
    raise RuntimeError("Another intake activity owns the registration mutex")


def release_registration_lock() -> None:
    DeltaTable.forName(spark_session, registration_leases_table).update(
        condition=(F.col("lock_name") == "global") & (F.col("owner_id") == event_key),
        set={"owner_id": F.lit(""), "expires_at": F.lit(datetime.now(timezone.utc))},
    )


existing_receipt = spark_session.table(receipts_table).where(F.col("event_key") == event_key).limit(2).collect()

if len(existing_receipt) > 1:
    release_registration_lock()
    raise RuntimeError(f"Duplicate event receipt rows exist for {event_key}")

if existing_receipt:
    outcome = existing_receipt[0].asDict(recursive=True)
    if outcome["registration_status"] == "REJECTED":
        release_registration_lock()
        raise ValueError(
            f"Previously rejected event: {outcome.get('error_type')}: "
            f"{outcome.get('error_message')}"
        )
else:
    try:
        manifest_text = notebookutils.fs.head(manifest_uri, 1024 * 1024)
        manifest_payload = json.loads(manifest_text)
        if not isinstance(manifest_payload, dict):
            raise ValueError("Manifest root must be a JSON object")
        manifest = parse_manifest(manifest_payload, manifest_uri)
        work_id = hashlib.sha256(
            f"{manifest['source_uri']}\n{manifest['asset_version']}".encode("utf-8")
        ).hexdigest()
        config_json, config_sha256 = canonical_config(manifest.pop("counting_line"))
        existing_work_rows = spark_session.table(work_table).where(F.col("work_id") == work_id).limit(2).collect()
        if len(existing_work_rows) > 1:
            raise RuntimeError(f"Duplicate work rows exist for {work_id}")
        preexisting_work = bool(existing_work_rows)
        if preexisting_work:
            existing_work = existing_work_rows[0].asDict(recursive=True)
            for field in (
                "asset_id",
                "asset_version",
                "source_uri",
                "manifest_uri",
                "source_etag",
                "expected_size_bytes",
                "expected_sha256",
                "camera_id",
                "location_id",
                "captured_at_utc",
                "camera_timezone",
                "duration_seconds",
                "config_sha256",
            ):
                expected = config_sha256 if field == "config_sha256" else manifest[field]
                actual = existing_work[field]
                if field == "captured_at_utc":
                    if actual.tzinfo is None:
                        actual = actual.replace(tzinfo=timezone.utc)
                    actual = actual.astimezone(timezone.utc)
                    expected = expected.astimezone(timezone.utc)
                if actual != expected:
                    raise ValueError(f"Existing work_id has conflicting {field}")
        work_row = {
            **manifest,
            "work_id": work_id,
            "priority": int(PRIORITY),
            "status": "QUEUED",
            "received_at": now,
            "queued_at": now,
            "not_before_at": None,
            "attempt_count": 0,
            "max_attempts": int(MAX_ATTEMPTS),
            "lease_owner_attempt_id": None,
            "lease_dispatcher_id": None,
            "lease_acquired_at": None,
            "lease_expires_at": None,
            "last_heartbeat_at": None,
            "committed_attempt_id": None,
            "completed_at": None,
            "last_error_category": None,
            "last_error_type": None,
            "last_error_message": None,
            "config_json": config_json,
            "config_sha256": config_sha256,
            "capture_date": manifest["captured_at_utc"].date(),
        }
        work_source = spark_session.createDataFrame([work_row], spark_session.table(work_table).schema)
        (
            DeltaTable.forName(spark_session, work_table)
            .alias("t")
            .merge(work_source.alias("s"), "t.work_id = s.work_id")
            .whenNotMatchedInsertAll()
            .execute()
        )
        registration_status = "EXISTING_WORK" if preexisting_work else "QUEUED"
        receipt_row = {
            "event_key": event_key,
            "event_source": event_source,
            "event_id": event_id,
            "event_type": event_type,
            "event_time": event_time,
            "subject": subject,
            "manifest_uri": manifest_uri,
            "work_id": work_id,
            "received_at": now,
            "registration_status": registration_status,
            "pipeline_run_id": PIPELINE_RUN_ID or None,
            "error_type": None,
            "error_message": None,
        }
        receipt_source = spark_session.createDataFrame([receipt_row], spark_session.table(receipts_table).schema)
        (
            DeltaTable.forName(spark_session, receipts_table)
            .alias("t")
            .merge(receipt_source.alias("s"), "t.event_key = s.event_key")
            .whenNotMatchedInsertAll()
            .execute()
        )
        outcome = {"event_key": event_key, "work_id": work_id, "status": registration_status}
    except ValueError as error:
        receipt_row = {
            "event_key": event_key,
            "event_source": event_source,
            "event_id": event_id,
            "event_type": event_type,
            "event_time": event_time,
            "subject": subject,
            "manifest_uri": manifest_uri,
            "work_id": None,
            "received_at": now,
            "registration_status": "REJECTED",
            "pipeline_run_id": PIPELINE_RUN_ID or None,
            "error_type": type(error).__name__,
            "error_message": str(error)[:4000],
        }
        rejected_source = spark_session.createDataFrame([receipt_row], spark_session.table(receipts_table).schema)
        (
            DeltaTable.forName(spark_session, receipts_table)
            .alias("t")
            .merge(rejected_source.alias("s"), "t.event_key = s.event_key")
            .whenNotMatchedInsertAll()
            .execute()
        )
        release_registration_lock()
        raise

release_registration_lock()
print(json.dumps(outcome, default=str, sort_keys=True))

In [ ]:
notebookutils.notebook.exit(json.dumps(outcome, default=str, sort_keys=True))